<div style="font-family: system-ui, -apple-system, sans-serif; text-align: center; padding: 48px 24px 24px;">
    <div style="display: inline-block; background: #be0f05; color: white;
                padding: 12px 20px; border-radius: 12px; margin-bottom: 20px;">
        <span style="font-size: 36px; font-weight: 800; letter-spacing: -0.5px;">TIP4PATLIBS &ndash; Regional Leads incl. National-Only Filers</span>
    </div>
    <div style="font-size: 16px; color: #475569; margin-bottom: 8px; line-height: 1.6;">
        The full regional applicant picture &mdash; <strong>PATSTAT for the EP/PCT-active firms</strong>, the <strong>DPMA register for the national-only tail</strong> PATSTAT can&rsquo;t geolocate.
    </div>
    <div style="font-size: 13px; color: #94a3b8; margin-bottom: 32px;">
        EPO Academy Training Material &nbsp;&middot;&nbsp; <a href="https://patentreports.depa.tech" target="_blank"
           style="color: #be0f05; text-decoration: none; font-weight: 600;">created by Arne Kr&uuml;ger</a>
    </div>
    <div style="background: #f8fafc; border-radius: 12px; padding: 24px 28px; max-width: 640px;
                margin: 0 auto; border: 1px solid #e2e8f0; text-align: left;">
        <div style="font-size: 14px; color: #334155; line-height: 1.9;">
            <strong>What you will do in this notebook</strong>
            <br/>Step&nbsp;1 &nbsp;&middot;&nbsp; Pull the <strong>DE company applicant population</strong> from PATSTAT (one year)
            <br/>Step&nbsp;2 &nbsp;&middot;&nbsp; Split into <strong>already-geolocated</strong> (NUTS in PATSTAT) vs <strong>national-only</strong>
            <br/>Step&nbsp;3 &nbsp;&middot;&nbsp; Resolve the national-only addresses via the <strong>DPMA register</strong> (AKZ lookup)
            <br/>Step&nbsp;4 &nbsp;&middot;&nbsp; Map PLZ &rarr; NUTS region
            <br/>Step&nbsp;5 &nbsp;&middot;&nbsp; <strong>Union</strong> &amp; build the regional lead list
            <br/>Step&nbsp;6 &nbsp;&middot;&nbsp; The payoff: what the PATSTAT-only view <strong>misses</strong>
        </div>
    </div>
    <div style="background: #fdf2f2; border-radius: 10px; padding: 16px 24px; max-width: 640px;
                margin: 28px auto 0; border: 1px solid #fecaca;">
        <div style="font-size: 14px; color: #404955; font-weight: 600;">&#9654; &nbsp;Run the cells top to bottom.</div>
        <div style="font-size: 12px; color: #64748b; margin-top: 6px; line-height: 1.6;">
            Needs the <strong>PATSTAT on EPO&nbsp;TIP</strong> (<code>PatstatClient(env=&#39;PROD&#39;)</code>) <strong>and</strong> DPMA credentials
            (<code>DPMA_USER</code>/<code>DPMA_PASS</code> in a <code>.env</code>). Bounded to one year and a resolve cap so it stays interactive;
            the full, cached country-wide mapping is the job of the PATSTAT-MCP extension table.
        </div>
    </div>
    <div style="margin-top: 20px; font-size: 12px; color: #cbd5e1;">
        Part of EPO TIP Working Group Sessions, 2026. &nbsp;Data: EPO PATSTAT Global, Autumn 2025 &nbsp;&middot;&nbsp; DPMAconnect&nbsp;Plus register &nbsp;&middot;&nbsp; Eurostat GISCO PLZ&rarr;NUTS.
    </div>
</div>

## Why this notebook exists

A NUTS region filter on PATSTAT returns only the **EP/PCT-active subset** of a region &mdash; national-only
German filings carry no NUTS. But PATSTAT still *contains* those applications (`appln_auth='DE'`); it just
lacks their region. So we pull the DE **company** applicant population from PATSTAT, and resolve **only the
national-only ones** (no NUTS) via the DPMA register &mdash; address is a company attribute, so it's one
lookup per firm, not per application.

Bounded here to **one filing year, patents + utility models, German companies**. Measured for 2023:
~2,600 German company applicants, ~2,100 already geolocated in PATSTAT, ~500 national-only.

In [ ]:
import sys, time
from pathlib import Path
import pandas as pd

# Make the dpma helper package importable, wherever Jupyter launched.
ROOT = next(c for c in [Path.cwd(), *Path.cwd().parents]
            if (c / "dpma" / "register_parser.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dpma import fetch, parse_hit_applicant, map_plz
from dpma.plz_nuts import BUNDESLAND_BY_NUTS1

# PATSTAT on EPO TIP (base conda env). sql_query runs BigQuery Standard SQL,
# so the queries below need no dialect changes.
from epo.tipdata.patstat import PatstatClient
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute a PATSTAT SQL query on TIP and return a pandas DataFrame."""
    start = time.time()
    df = pd.DataFrame(patstat.sql_query(query, use_legacy_sql=False))
    print(f"Query took {time.time() - start:.1f}s ({len(df)} rows)")
    return df

pd.set_option("display.max_rows", 60)
print("Connected to PATSTAT (TIP, env='PROD') + DPMA helpers loaded.")

In [ ]:
# --- DPMA credentials from a local .env file (never commit .env) -------------
import os
def load_env(filename=".env"):
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / filename
        if f.exists():
            for line in f.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return f
    return None
print("Loaded .env from:", load_env() or "(none found)")
print("DPMA_USER set:", bool(os.environ.get("DPMA_USER")), "| DPMA_PASS set:", bool(os.environ.get("DPMA_PASS")))

## Step 1 &mdash; the DE company applicant population (PATSTAT)

One query, aggregated per harmonised applicant (`psn_id`): filing count (`n_appln` = depth), whether the
firm has any EP/PCT family (`has_ep`), whether PATSTAT already has a NUTS region for it (`has_nuts`), a
representative application number to resolve (`repr_appln_nr`), and the PATSTAT NUTS if present.

In [ ]:
YEAR  = 2023
KINDS = "('A','U')"     # A = patents, U = utility models

df = run_query(f"""
WITH de AS (
  SELECT a.appln_id, a.appln_nr, TRIM(a.appln_kind) AS kind, a.docdb_family_id
  FROM tls201_appln a
  WHERE a.appln_auth='DE' AND TRIM(a.appln_kind) IN {KINDS} AND a.appln_filing_year={YEAR}
),
epfam AS (   -- families that also have an EP/PCT member
  SELECT DISTINCT docdb_family_id FROM tls201_appln
  WHERE appln_auth IN ('EP','WO') AND docdb_family_id IN (SELECT docdb_family_id FROM de)
),
app AS (     -- German company applicants only
  SELECT de.appln_id, de.appln_nr, de.kind, p.psn_id, p.psn_name,
         (epfam.docdb_family_id IS NOT NULL) AS has_ep
  FROM de
  JOIN tls207_pers_appln pa ON pa.appln_id=de.appln_id AND pa.applt_seq_nr>0
  JOIN tls206_person p ON p.person_id=pa.person_id
  LEFT JOIN epfam ON epfam.docdb_family_id=de.docdb_family_id
  WHERE p.psn_sector='COMPANY' AND p.person_ctry_code='DE'
),
nutsflag AS ( -- does this applicant have ANY German NUTS in PATSTAT?
  SELECT psn_id,
         MAX(CASE WHEN nuts LIKE 'DE%' AND nuts_level>=3 THEN 1 ELSE 0 END) AS has_nuts,
         MAX(CASE WHEN nuts LIKE 'DE%' AND nuts_level>=3 THEN nuts END) AS patstat_nuts
  FROM tls206_person
  WHERE psn_id IN (SELECT DISTINCT psn_id FROM app)
  GROUP BY psn_id
)
SELECT app.psn_id,
       ANY_VALUE(app.psn_name) AS psn_name,
       COUNT(DISTINCT app.appln_id) AS n_appln,
       MAX(CAST(app.has_ep AS INT64)) AS has_ep,
       MAX(nf.has_nuts) AS has_nuts,
       ANY_VALUE(nf.patstat_nuts) AS patstat_nuts,
       ANY_VALUE(app.appln_nr) AS repr_appln_nr
FROM app LEFT JOIN nutsflag nf USING(psn_id)
GROUP BY app.psn_id
ORDER BY n_appln DESC
""")

print(f"{len(df)} German company applicants in {YEAR} | "
      f"{int((df.has_nuts==1).sum())} geolocated in PATSTAT | "
      f"{int((df.has_nuts==0).sum())} national-only")
df.head()

## Step 2 &mdash; split: who already has a region, who needs DPMA

In [ ]:
geo = df[df.has_nuts == 1].copy()                                   # region already in PATSTAT
nat = df[df.has_nuts == 0].copy().sort_values("n_appln", ascending=False)  # national-only -> DPMA
print(f"{len(geo)} already geolocated (free)  |  {len(nat)} national-only (need DPMA)")

## Step 3 &mdash; resolve the national-only addresses via the DPMA register

For each national-only firm we look up **one** of its applications by Aktenzeichen (`search/AKZ=<appln_nr>`,
which accepts the raw PATSTAT number) and read the applicant's PLZ straight from the hit. Capped at
`MAX_RESOLVE` for interactivity &mdash; a small fraction won't resolve (city-only hits, ownership changes,
foreign co-applicants) and are skipped.

In [ ]:
MAX_RESOLVE = 200      # raise to len(nat) for full coverage (~500 for 2023)

client = fetch.DpmaClient()          # reads DPMA_USER / DPMA_PASS
todo = nat.head(MAX_RESOLVE)
rows = []
for i, r in enumerate(todo.itertuples(), 1):
    a = {}
    try:
        hits = client.search_aktenzeichen(r.repr_appln_nr)
        if hits:
            a = parse_hit_applicant(hits[0].applicants[0])
    except Exception:
        pass
    rows.append({"psn_id": r.psn_id, "psn_name": r.psn_name, "n_appln": r.n_appln,
                 "has_ep": r.has_ep, "plz": a.get("plz"), "city": a.get("city"),
                 "country": a.get("country")})
    if i % 50 == 0:
        print(f"  resolved {i}/{len(todo)}")

res = pd.DataFrame(rows)
print(f"resolved a German PLZ for {int(res.plz.notna().sum())} of {len(res)} firms")

## Step 4 &mdash; PLZ &rarr; NUTS region

In [ ]:
def to_region(plz):
    m = map_plz(plz)
    return pd.Series([m["nuts3"], m["bundesland"]] if m else [None, None])

res[["nuts3", "bundesland"]] = res["plz"].apply(to_region)
res_de = res.dropna(subset=["nuts3"]).copy()
res_de["source"] = "dpma-national-only"
print(f"{len(res_de)} national-only firms placed into a German region")

## Step 5 &mdash; union &amp; the regional lead list

Combine the PATSTAT-geolocated firms (region from their NUTS) with the DPMA-resolved national-only firms,
then filter to one region. Change `REGION` to any Bundesland.

In [ ]:
# PATSTAT-geolocated firms -> region from their NUTS3 (first 5 chars of the code)
geo2 = geo.dropna(subset=["patstat_nuts"]).copy()
geo2["nuts3"] = geo2["patstat_nuts"].str[:5]
geo2["bundesland"] = geo2["nuts3"].str[:3].map(BUNDESLAND_BY_NUTS1)
geo2["source"] = "patstat-nuts"
geo2["plz"] = None
geo2["city"] = None

cols = ["psn_name", "plz", "city", "nuts3", "bundesland", "n_appln", "has_ep", "source"]
full = pd.concat([geo2[cols], res_de[cols]], ignore_index=True)

REGION = "Bayern"
leads = full[full["bundesland"] == REGION].sort_values("n_appln", ascending=False)
leads.head(40)

## Step 6 &mdash; the payoff: what the PATSTAT-only view misses

In [ ]:
n_total = len(leads)
n_nat   = int((leads.source == "dpma-national-only").sum())
share   = n_nat / n_total * 100 if n_total else 0
print(f"{REGION}, {YEAR}: {n_total} company leads")
print(f"  {n_total - n_nat} visible to the PATSTAT-only notebook")
print(f"  {n_nat} national-only -> recovered ONLY via the DPMA register ({share:.0f}% of the region's leads)")
print(f"(national-only capped at MAX_RESOLVE={MAX_RESOLVE}; raise it for the full tail)")

## Caveats

- **Bounded demo:** one year, `MAX_RESOLVE` cap. The full, country-wide, cached mapping belongs in the
  **PATSTAT-MCP extension table** (`de_applicant_nuts`) so any query can join it &mdash; see the hand-off brief.
- **Foreign filers** at the DPMA (no German PLZ) are dropped from the regional view by design.
- **Address is a company attribute:** one DPMA lookup per `psn_id`, reusable across all its filings and years.
- **GDPR:** companies only (`psn_sector='COMPANY'`); natural-person applicants are excluded.
- **NUTS vintage:** PATSTAT NUTS vs the Eurostat 2024 crosswalk can differ for a few recoded regions; align
  vintages before a strict code-level comparison.